# Module 01 — Sentinel-2 Xarray Pipeline for Bathymetry Estimation

**Workflow overview (with ACOLITE DSF)**

```
Full S2 scene (~100×100 km)
    └─► select_sentinel2_bands → to_rrs → to_10m
            └─► ACOLITE DSF on full scene  ← dark pixels found here
                    └─► deglint
                            └─► .clip(aoi_geom)  ← clip AFTER correction
                                    └─► stack → .mean() composite
                                            └─► sampleRegions at points
```

## INSTALLATION AND IMPORT

In [ ]:
!pip install -q pystac_client stackstac planetary_computer xarray dask[array]
!pip install -q xee geopandas   # Google Earth Engine Xarray backend

# PINN model dependencies
!pip install -q tensorflow scikit-learn rasterio rio-cogeo netcdf4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.3/64.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.3/479.3 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.0 MB/s eta 0:00:00


In [ ]:
## Connect to ACOLITE version and import (takes aroud 1 minute)
import sys
ACOLITE_PATH = '/content/drive/MyDrive/Bathymetry/Notebook/acolite-main/acolite-main' # ACOLITE version itself
sys.path.append(ACOLITE_PATH)
import ACOLITE

In [ ]:
import os
import ee
import time
import geemap
import sqlite3
import stackstac
import pystac_client
import planetary_computer

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt

from rasterio.transform import from_bounds
from rasterio.crs      import CRS
from rio_cogeo.cogeo    import cog_translate
from rio_cogeo.profiles import cog_profiles

from tqdm import tqdm
from datetime import datetime
from pathlib import Path
from pystac_client import Client

import torch
import torch.nn as nn

import sys
sys.path.insert(0, '/content/drive/MyDrive/Bathymetry/Notebook')

from bathymetry_model import (
    load_bathymetry_model,
    BathymetryInference,
    EXPECTED_BANDS
)

from xee.ext import EarthEngineBackendEntrypoint
import xarray.backends.plugins as xr_plugins

xr_plugins.BACKEND_ENTRYPOINTS['ee'] = ('xee.ext', EarthEngineBackendEntrypoint)

import importlib
import sys

# Remove cached module so Python re-reads the file from disk
if 'bathymetry_model' in sys.modules:
    del sys.modules['bathymetry_model']

from bathymetry_model import load_bathymetry_model, BathymetryInference, EXPECTED_BANDS
print('✓ bathymetry_model reloaded')

# Also delete the .pyc cache to be sure
import pathlib
for pyc in pathlib.Path('/content/drive/MyDrive/Bathymetry/Notebook').glob('bathymetry_model*.pyc'):
    pyc.unlink()
for pyc in pathlib.Path('/content/drive/MyDrive/Bathymetry/Notebook/__pycache__').glob('bathymetry_model*.pyc'):
    pyc.unlink()

✓ bathymetry_model reloaded


## INITIALIZE EARTH ENGINE

In [ ]:
ee.Authenticate()
ee.Initialize(project='ee-chriscandido93')
print('GEE initialised successfully')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

GEE initialised successfully
Using device: cpu


## LOAD TILES

In [ ]:
# =========================
# PATHS
# =========================

gpkg_dir   = '/content/drive/MyDrive/Bathymetry/ECHOSOUNDER/2024'
output_dir = '/content/drive/MyDrive/Bathymetry/ECHOSOUNDER/bathymetry'
drive_folder = 'Bathymetry/ECHOSOUNDER/bathymetry'  # folder in Google Drive for exports
os.makedirs(output_dir, exist_ok=True)

# =========================
# COMPUTE BOUNDING BOX PER GPKG FILE
# =========================
# Bounds derived from actual point locations — used to scope the S2 search
# and the AOI clip after ACOLITE correction.

gpkg_files = sorted(f for f in os.listdir(gpkg_dir) if f.endswith('.gpkg'))

bounds_list = []
for fname in gpkg_files:
    gdf = gpd.read_file(os.path.join(gpkg_dir, fname))
    gdf.columns = [c.lower() for c in gdf.columns]
    if 'lon' not in gdf.columns and gdf.geometry is not None:
        gdf['lon'] = gdf.geometry.x
        gdf['lat'] = gdf.geometry.y
    bounds_list.append({
        'filename': fname,
        'min_lon' : float(gdf['lon'].min()),
        'min_lat' : float(gdf['lat'].min()),
        'max_lon' : float(gdf['lon'].max()),
        'max_lat' : float(gdf['lat'].max()),
        'n_points': len(gdf),
    })

bounds_df = pd.DataFrame(bounds_list)
print(bounds_df.to_string(index=False))

                        filename    min_lon   min_lat    max_lon   max_lat  n_points
bathymetry_masinloc_dry2024.gpkg 119.899192 15.504893 119.946725 15.551578      1356
   bathymetry_nogas_dry2024.gpkg 121.912823 10.413298 121.924071 10.424595      2377
 bathymetry_stacruz_dry2024.gpkg 121.069075  6.857627 122.075330 14.656900     13493


In [ ]:
# =========================
# TIME RANGE
# =========================

start_date = '2025-01-01'
end_date   = '2025-06-30'

print(f'Time range : {start_date} → {end_date}')
print(f'Files      : {len(bounds_df)}')


Time range : 2025-01-01 → 2025-06-30
Files      : 3


## DEFINE FUNCTIONS

In [ ]:
# =========================
# ACOLITE PREPROCESSING HELPERS (GEE-side)
# =========================
# select_sentinel2_bands -> to_rrs -> to_10m -> dask_spectrum_fitting -> deglint_alternative

def correct_collection_with_acolite(collection):
    # Step 1: band selection, Rrs conversion, 10m resampling (mappable)
    collection_rrs = (
        collection
        .map(ACOLITE.select_sentinel2_bands)
        .map(ACOLITE.to_rrs)
        .map(ACOLITE.to_10m)
    )

    # DSF + deglint
    collection_list = collection_rrs.toList(collection_rrs.size().getInfo())
    n = collection_list.size().getInfo()

    corrected = []
    for i in range(n):
        img = ee.Image(collection_list.get(i))
        # dask_spectrum_fitting returns (corrected_image, metadata_dict)
        img_dsf = ACOLITE.dask_spectrum_fitting(img)
        # deglint_alternative takes the same (img, meta) as *args
        img_deglint = ACOLITE.deglint_alternative(*img_dsf)
        corrected.append(img_deglint)

    return ee.ImageCollection.fromImages(corrected)


def add_water_quality_products(collection):
    """Add Chl-a and SPM bands to an ACOLITE-corrected collection."""
    collection = collection.map(ACOLITE.add_chl_re_mishra)
    collection = collection.map(ACOLITE.add_spm_nechad2016_665)
    return collection


print('\u2713 ACOLITE preprocessing helpers defined')

✓ ACOLITE preprocessing helpers defined


In [ ]:
# =========================
# CLOUD MASKING THRESHOLDS
# =========================

MAX_CLOUD_PROBABILITY = 30
MAX_CLOUD_SCORE_PLUS  = 0.5


def mask_edges(image):
    """Mask scene edges where 20 m / 60 m bands have no data."""
    return image.updateMask(
        image.select('B8A').mask().updateMask(image.select('B9').mask())
    )


def mask_clouds_s2cloudless(image):
    """Mask clouds via s2cloudless probability band (joined as 'cloud_mask')."""
    cloud_prob = ee.Image(image.get('cloud_mask')).select('probability')
    return image.updateMask(cloud_prob.lt(MAX_CLOUD_PROBABILITY))


def mask_clouds_csp(image):
    """Mask clouds via Cloud Score+ cs_cdf band (joined as 'csp_mask')."""
    csp_mask = image.get('csp_mask')
    cs = ee.Algorithms.If(
        ee.Algorithms.IsEqual(csp_mask, None),
        ee.Image.constant(1),
        ee.Image(csp_mask).select('cs_cdf')
    )
    return image.updateMask(ee.Image(cs).gte(MAX_CLOUD_SCORE_PLUS))


def mask_shadows(image):
    """Mask cloud shadows via directional distance transform."""
    nir   = image.select('B8')
    swir1 = image.select('B11')
    blue  = image.select('B2')

    is_dark      = nir.lt(0.12)
    is_not_water = swir1.gt(0.015).Or(blue.divide(nir.add(0.001)).lt(1.5))
    potential_shadow = is_dark.And(is_not_water)

    cloud_mask_img = image.get('cloud_mask')
    cloud_for_proj = ee.Algorithms.If(
        ee.Algorithms.IsEqual(cloud_mask_img, None),
        ee.Image(image.get('csp_mask')).select('cs_cdf').lt(MAX_CLOUD_SCORE_PLUS).unmask(0),
        ee.Image(cloud_mask_img).select('probability').gt(MAX_CLOUD_PROBABILITY)
    )
    cloud_mask = ee.Image(cloud_for_proj)

    shadow_azimuth = ee.Number(90).subtract(
        ee.Number(image.get('MEAN_SOLAR_AZIMUTH_ANGLE'))
    )
    cloud_proj = (
        cloud_mask
        .directionalDistanceTransform(shadow_azimuth, 10)
        .reproject(crs=image.select('B2').projection(), scale=30)
        .select('distance')
        .mask()
    )
    return image.updateMask(potential_shadow.And(cloud_proj).Not())


def mask_clouds_and_quality(image):
    """
    Combined pipeline: s2cloudless -> Cloud Score+ -> shadow -> scale to [0,1].
    """
    masked = mask_clouds_s2cloudless(image)
    masked = mask_clouds_csp(masked)
    masked = mask_shadows(masked)
    return (
        masked
        .divide(10000)
        .copyProperties(image, ['system:time_start', 'CLOUDY_PIXEL_PERCENTAGE'])
    )


def mask_to_extent(image):
    """Clip to AOI extent."""
    mask = ee.Image.constant(1).clip(aoi).mask()
    return image.updateMask(mask)


def mask_to_reef(image):
    """Restrict to Allen Coral Atlas reef pixels."""
    aca = ee.Image('ACA/reef_habitat/v2_0').select('reef_mask')
    return image.updateMask(aca.eq(1)).copyProperties(image, ['system:time_start'])


print('\u2713 Cloud-masking helpers defined')

✓ Cloud-masking helpers defined


In [ ]:
# =========================
# GET CLEANED L1C COLLECTION (per tile) — ACOLITE input
# =========================
# Uses COPERNICUS/S2_HARMONIZED (L1C TOA) — ACOLITE DSF requires L1C input.
# Cloud/shadow masking is applied before ACOLITE

# Bands to keep for downstream PINN (after ACOLITE renames to Rrs_*)
BANDS = ['B1', 'B2', 'B3', 'B4', 'B8', 'B8A', 'B11', 'B12']  # PINN input bands

# ACOLITE Rrs band name -> PINN band name mapping
ACOLITE_TO_PINN = {
    'Rrs_443': 'B1',
    'Rrs_492': 'B2',
    'Rrs_560': 'B3',
    'Rrs_665': 'B4',
    'Rrs_833': 'B8',
    'Rrs_865': 'B8A',
    'Rrs_1614': 'B11',
    'Rrs_2202': 'B12',
}


def get_cleaned_collection_l1c(tile_geom, start_date, end_date, max_cloud_cover=30):
    criteria = ee.Filter.And(
        ee.Filter.bounds(tile_geom),
        ee.Filter.date(start_date, end_date)
    )

    s2_l1c = (
        ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
        .filter(criteria)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud_cover))
        .map(mask_edges)
    )

    # Join 1 — s2cloudless
    s2_clouds = ee.ImageCollection('COPERNICUS/S2_CLOUD_PROBABILITY').filter(criteria)
    s2_joined = ee.ImageCollection(
        ee.Join.saveFirst('cloud_mask').apply(
            primary=s2_l1c, secondary=s2_clouds,
            condition=ee.Filter.equals(leftField='system:index', rightField='system:index')
        )
    ).filter(ee.Filter.notNull(['cloud_mask']))

    # Join 2 — Cloud Score+
    s2_csp = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED').filter(criteria)
    s2_dual = ee.ImageCollection(
        ee.Join.saveFirst('csp_mask').apply(
            primary=s2_joined, secondary=s2_csp,
            condition=ee.Filter.equals(leftField='system:index', rightField='system:index')
        )
    )

    # Apply cloud/shadow masks
    cleaned = (
        s2_dual
        .map(mask_clouds_s2cloudless)
        .map(mask_clouds_csp)
        .map(mask_shadows)
        .map(mask_to_extent)
    )
    # Copy time property for downstream joins
    cleaned = cleaned.map(
        lambda img: img.copyProperties(img, ['system:time_start', 'CLOUDY_PIXEL_PERCENTAGE'])
    )

    return cleaned


print('\u2713 get_cleaned_collection_l1c defined')

✓ get_cleaned_collection_l1c defined


In [ ]:
# =========================
# ACOLITE DSF PREPROCESSING
# =========================
# Strategy: run ACOLITE DSF on the FULL S2 scene so it has enough
# valid dark pixels for percentile-based dark-spectrum estimation.
# Clip to the survey AOI only after correction, then composite.

def get_l1c_collection(aoi_geom, start_date, end_date, max_cloud_cover=30):
    """
    Return full (unclipped) S2 L1C scenes intersecting the AOI,
    filtered by date and scene-level cloud cover only.
    """
    return (
        ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
        .filterDate(start_date, end_date)
        .filterBounds(aoi_geom)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_cloud_cover))
    )


def correct_scene_with_acolite(img):
    """
    Run ACOLITE DSF + deglint on a single full S2 L1C scene.
    Returns the corrected Rrs ee.Image, or None on failure.
    """
    try:
        img_rrs     = ACOLITE.to_10m(
                          ACOLITE.to_rrs(
                              ACOLITE.select_sentinel2_bands(img)))
        dsf_result  = ACOLITE.dask_spectrum_fitting(img_rrs)
        img_deglint = ACOLITE.deglint_alternative(*dsf_result)
        return img_deglint
    except Exception as e:
        return None


def build_rrs_composite(l1c_collection, aoi_geom):
    """
    For each scene in l1c_collection:
      1. Run ACOLITE DSF on the full unclipped scene
      2. Clip the corrected Rrs image to aoi_geom
    Then take the temporal mean across all successful scenes.

    Returns:
        (ee.Image, int) — mean Rrs composite clipped to AOI, n scenes used
        (None, 0)       — if all scenes failed DSF
    """
    scene_list = l1c_collection.toList(l1c_collection.size().getInfo())
    n          = scene_list.size().getInfo()

    corrected = []
    for i in range(n):
        img    = ee.Image(scene_list.get(i))
        result = correct_scene_with_acolite(img)
        if result is None:
            print(f'    scene {i+1}/{n}: ⚠  DSF failed — skipped')
            continue
        corrected.append(result.clip(aoi_geom))
        print(f'    scene {i+1}/{n}: ✓ DSF done')

    if not corrected:
        return None, 0

    composite = ee.ImageCollection.fromImages(corrected).map(lambda img : img.float()).mean()
    return composite, len(corrected)


print('✓ ACOLITE functions defined')

✓ ACOLITE functions defined


In [ ]:
import xarray as xr
import numpy  as np
import pandas as pd

RRS_BANDS = ['B1', 'B2', 'B3', 'B4', 'B8', 'B8A', 'B11', 'B12']

def composite_to_xarray(composite, aoi_geom, rrs_bands, scale=0.0001):
    """
    Pull an ee.Image composite into an xarray.Dataset via xee.

    The composite must already be clipped to aoi_geom (as built by
    build_rrs_composite). xee opens it lazily; .compute() triggers
    the actual download — only the AOI pixels, not the full S2 tile.

    Args:
        composite  : ee.Image  — mean Rrs composite, clipped to AOI
        aoi_geom   : ee.Geometry.Rectangle — used to set the region
        rrs_bands  : list[str] — band names to pull (e.g. ['B1','B2',...])
        scale      : int — pixel resolution in metres (default 10)

    Returns:
        xarray.Dataset with dims (lat, lon) and one var per band
    """
    bounds   = aoi_geom.bounds().getInfo()['coordinates'][0]
    lons     = [c[0] for c in bounds]
    lats     = [c[1] for c in bounds]
    bbox     = [min(lons), min(lats), max(lons), max(lats)]   # [W, S, E, N]

    ds = xr.open_dataset(
        composite.select(rrs_bands),
        engine        = 'ee',
        geometry      = ee.Geometry.BBox(*bbox),
        scale         = scale,
        crs           = 'EPSG:4326',
        chunks        = {'time': -1},          # enable dask
    )

    print(f'    Downloading composite ({len(rrs_bands)} bands @ {scale}m)...')
    ds = ds.compute()               # pull from GEE into memory
    ds = ds.squeeze('time', drop=True)
    print(f'    Done. Grid: {dict(ds.dims)}')
    return ds


def extract_points_from_xarray(ds, gdf, rrs_bands):
    """
    Nearest-neighbour point extraction from an xarray.Dataset.

    Args:
        ds        : xarray.Dataset with (lat, lon) dims — from composite_to_xarray
        gdf       : GeoDataFrame with 'lat', 'lon', 'depth', 'wtemp' columns
        rrs_bands : list[str] — band names present in ds

    Returns:
        pd.DataFrame with columns [row_id, lat, lon, depth, wtemp, B1, B2, ...]
    """
    lats = xr.DataArray(gdf['lat'].values, dims='points')
    lons = xr.DataArray(gdf['lon'].values, dims='points')

    # sel with method='nearest' does vectorised nearest-neighbour lookup
    sampled = ds[rrs_bands].sel(lat=lats, lon=lons, method='nearest')

    result = gdf[['lat', 'lon']].copy().reset_index(drop=False).rename(
        columns={'index': 'row_id'}
    )

    # pull optional columns safely
    for col in ('depth', 'wtemp'):
        if col in gdf.columns:
            result[col] = gdf[col].values

    for band in rrs_bands:
        result[band] = sampled[band].values

    return result


def process_all_gpkg_xarray(gpkg_dir, bounds_df, start_date, end_date,
                             rrs_bands=None,
                             max_cloud_cover=30,
                             scale=10,
                             output_dir=output_dir):
    """
    Full pipeline — xarray edition (no GEE Export, no Drive, no payload limit):
      1. Build AOI from point bounding box
      2. Fetch full S2 L1C scenes → ACOLITE DSF → clip → mean composite  (GEE-side)
      3. Pull composite into xarray via xee                               (download)
      4. Extract band values at echosounder points via nearest-neighbour  (local)
      5. Save to CSV in output_dir

    Output: one CSV per GPKG, named extracted_<stem>.csv
    """
    if rrs_bands is None:
        rrs_bands = RRS_BANDS

    print(f"\n{'='*70}")
    print(f'ECHOSOUNDER EXTRACTION — xarray edition — {len(bounds_df)} files')
    print(f'Bands      : {rrs_bands}')
    print(f'Output dir : {output_dir}')
    print(f"{'='*70}")

    for _, row in bounds_df.iterrows():
        fname  = row['filename']
        fpath  = os.path.join(gpkg_dir, fname)
        stem   = fname.replace('.gpkg', '')
        outcsv = os.path.join(output_dir, f'extracted_{stem}.csv')

        print(f"\n{'─'*70}")
        print(f'  {fname}  ({row["n_points"]} points)')

        # ── AOI ───────────────────────────────────────────────────────────
        buf      = 0.01
        aoi_geom = ee.Geometry.Rectangle(
            [row['min_lon'] - buf, row['min_lat'] - buf,
             row['max_lon'] + buf, row['max_lat'] + buf],
            proj='EPSG:4326', geodesic=False,
        )

        # ── Build composite (GEE-side, same as before) ────────────────────
        l1c      = get_l1c_collection(aoi_geom, start_date, end_date, max_cloud_cover)
        n_scenes = l1c.size().getInfo()
        if n_scenes == 0:
            print(f'  ✗ No valid scenes — skipping')
            continue
        print(f'  {n_scenes} scenes — running ACOLITE DSF...')

        composite, n_ok = build_rrs_composite(l1c, aoi_geom)
        if composite is None:
            print(f'  ✗ All scenes failed DSF — skipping')
            continue
        print(f'  Composite: {n_ok}/{n_scenes} scenes ✓')

        # ── Pull composite into xarray (replaces Export.table.toDrive) ────
        ds = composite_to_xarray(composite, aoi_geom, rrs_bands, scale=scale)

        # ── Load echosounder points ────────────────────────────────────────
        gdf = gpd.read_file(fpath)
        gdf.columns = [c.lower() for c in gdf.columns]
        if 'lon' not in gdf.columns and gdf.geometry is not None:
            gdf['lon'] = gdf.geometry.x
            gdf['lat'] = gdf.geometry.y
        gdf = gdf.reset_index(drop=True)

        # ── Extract band values at points (fully local) ───────────────────
        print(f'  Extracting {len(gdf)} points...')
        result_df = extract_points_from_xarray(ds, gdf, rrs_bands)

        # ── Save ──────────────────────────────────────────────────────────
        result_df.to_csv(outcsv, index=False)
        print(f'  ✓ Saved → {outcsv}  ({len(result_df)} rows)')

        ds.close()

    print(f"\n{'='*70}")
    print('All files processed.')
    print(f"{'='*70}")


# =========================
# RUN
# =========================

process_all_gpkg_xarray(
    gpkg_dir        = gpkg_dir,
    bounds_df       = bounds_df,
    start_date      = start_date,
    end_date        = end_date,
    rrs_bands       = RRS_BANDS,
    max_cloud_cover = 30,
    scale           = 0.0001,
    output_dir      = output_dir,
)


ECHOSOUNDER EXTRACTION — xarray edition — 1 files
Bands      : ['B1', 'B2', 'B3', 'B4', 'B8', 'B8A', 'B11', 'B12']
Output dir : /content/drive/MyDrive/Bathymetry/ECHOSOUNDER/bathymetry

──────────────────────────────────────────────────────────────────────
  bathymetry_stacruz_dry2024.gpkg  (13443 points)
  9 scenes — running ACOLITE DSF...
<---------------------------------------------------------------------------------------------------->
Pdark: {'B1': 0.08420000000000004, 'B2': 0.0383, 'B3': 0.0164, 'B4': 0, 'B5': 0, 'B6': 0, 'B7': 0, 'B8': 0, 'B8A': 0, 'B9': 0, 'B10': 9.999999999999994e-05, 'B11': 0.0005, 'B12': 0}
Geometry: sza=37.1170528622299, vza=4.043551158581447, raa=34.63197707869769
Gases: uwv=1.5, uoz=0.3, pressure=1013.25
Lut: sel_lut='ACOLITE-LUT-202110-MOD1', sel_aot=np.float64(0.001)
<---------------------------------------------------------------------------------------------------->
    scene 1/9: ✓ DSF done
<--------------------------------------------------------

/usr/local/lib/python3.12/dist-packages/xee/ext.py:696: UserWarning: Unable to retrieve 'system:time_start' values from an ImageCollection due to: No 'system:time_start' values found in the 'ImageCollection'.
  warnings.warn(


/tmp/ipykernel_17307/1669574509.py:41: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f'    Done. Grid: {dict(ds.dims)}')


    Done. Grid: {'lon': 494, 'lat': 358}
  Extracting 13443 points...
  ✓ Saved → /content/drive/MyDrive/Bathymetry/ECHOSOUNDER/bathymetry/extracted_bathymetry_stacruz_dry2024.csv  (13443 rows)

All files processed.


In [ ]:
# =========================
# SAMPLING HELPERS
# =========================
# Root cause of the 10 MB EEException:
#   ee.FeatureCollection built from thousands of inline points is serialised
#   into the Export.table.toDrive request body, which exceeds GEE's 10 MB
#   API payload limit. Fix: split points into chunks (≤500 per task) so each
#   serialised FC stays well under the limit.  All chunk-tasks for a file are
#   collected and monitored together; the caller merges the resulting CSVs.

POINT_CHUNK_SIZE = 100   # tune down if you still hit the limit


def gpkg_to_ee_fc(gdf):
    """Convert a GeoDataFrame (or slice) to an ee.FeatureCollection."""
    features = []
    for idx, row in gdf.iterrows():
        props = {
            'row_id': int(idx),
            'lat'   : float(row['lat']),
            'lon'   : float(row['lon']),
        }
        if pd.notna(row.get('depth')):
            props['depth'] = float(row['depth'])
        if pd.notna(row.get('wtemp')):
            props['wtemp'] = float(row['wtemp'])
        features.append(
            ee.Feature(ee.Geometry.Point([row['lon'], row['lat']]), props)
        )
    return ee.FeatureCollection(features)


def start_export_task(composite, points_fc, rrs_bands,
                      export_name, drive_folder, scale=10):
    """
    Start a single async GEE Export.table.toDrive task for one FC chunk.

    Args:
        composite    : ee.Image  — mean Rrs composite
        points_fc    : ee.FeatureCollection — echosounder points (one chunk)
        rrs_bands    : list[str] — band names to sample
        export_name  : str       — task description / output filename prefix
        drive_folder : str       — Google Drive destination folder
        scale        : int       — sampling resolution in metres

    Returns:
        ee.batch.Task
    """
    selectors = ['row_id', 'lat', 'lon', 'depth', 'wtemp'] + rrs_bands

    sampled = composite.select(rrs_bands).sampleRegions(
        collection=points_fc,
        scale=scale,
        geometries=False,
        tileScale=16,
    )

    task = ee.batch.Export.table.toDrive(
        collection     = sampled,
        description    = export_name,
        fileNamePrefix = export_name,
        fileFormat     = 'CSV',
        folder         = drive_folder,
        selectors      = selectors,
    )
    task.start()
    return task


def start_chunked_export_tasks(composite, gdf, rrs_bands,
                                base_name, drive_folder,
                                scale=10, chunk_size=POINT_CHUNK_SIZE):
    """
    Split *gdf* into chunks of *chunk_size* rows and submit one export task
    per chunk.  Returns a list of task-dicts compatible with monitor_tasks().

    Chunk CSVs are named  <base_name>_part<N>.csv  (zero-padded).
    If the GDF fits in a single chunk the name is just <base_name>.csv.
    """
    chunks = [gdf.iloc[i:i + chunk_size] for i in range(0, len(gdf), chunk_size)]
    n_chunks = len(chunks)
    tasks = []

    for k, chunk_df in enumerate(chunks):
        name = base_name if n_chunks == 1 else f'{base_name}_part{k:03d}'
        fc   = gpkg_to_ee_fc(chunk_df)
        task = start_export_task(
            composite    = composite,
            points_fc    = fc,
            rrs_bands    = rrs_bands,
            export_name  = name,
            drive_folder = drive_folder,
            scale        = scale,
        )
        tasks.append({'name': name, 'task': task})
        print(f'    chunk {k+1}/{n_chunks}: {len(chunk_df)} pts → {name}.csv ✓')

    return tasks


def monitor_tasks(tasks, check_interval=30):
    """
    Poll all export tasks until every one has completed or failed.
    Press Ctrl+C to stop monitoring (tasks keep running on GEE servers).
    """
    print(f"\n{'='*70}")
    print(f'MONITORING {len(tasks)} EXPORT TASK(S)')
    print(f'Check: https://code.earthengine.google.com/tasks')
    print(f"{'='*70}")

    try:
        while True:
            statuses = {t['name']: t['task'].status()['state'] for t in tasks}
            for name, state in statuses.items():
                icon = {'COMPLETED': '✅', 'RUNNING': '⟳',
                        'FAILED': '✗', 'PENDING': '○'}.get(state, '?')
                print(f'  {icon} {name:55s} {state}')

            if all(s in ('COMPLETED', 'FAILED') for s in statuses.values()):
                n_ok   = sum(1 for s in statuses.values() if s == 'COMPLETED')
                n_fail = sum(1 for s in statuses.values() if s == 'FAILED')
                print(f"\n✅ Done — {n_ok} completed, {n_fail} failed")
                break

            print(f'  ... checking again in {check_interval}s ...\n')
            time.sleep(check_interval)

    except KeyboardInterrupt:
        print('\nMonitoring stopped. Tasks continue on GEE servers.')
        print(f'Check: https://code.earthengine.google.com/tasks')


print('\u2713 Sampling helpers defined (chunked export, POINT_CHUNK_SIZE={})'.format(POINT_CHUNK_SIZE))


✓ Sampling helpers defined (chunked export, POINT_CHUNK_SIZE=500)


## ECHOSOUNDER DATA EXTRACTION

In [ ]:
# =========================
# EXTRACTION PIPELINE
# =========================

RRS_BANDS = ['B1', 'B2', 'B3', 'B4', 'B8', 'B8A', 'B11', 'B12']


def process_all_gpkg(gpkg_dir, bounds_df, start_date, end_date,
                     rrs_bands=RRS_BANDS,
                     max_cloud_cover=30,
                     scale=10,
                     drive_folder=drive_folder,
                     monitor=True,
                     check_interval=30):
    """
    For each echosounder GPKG file:
      1. Build AOI from point bounding box
      2. Fetch full S2 L1C scenes intersecting AOI
      3. ACOLITE DSF on each full scene → clip to AOI → mean composite
      4. Split points into chunks of POINT_CHUNK_SIZE and start one
         Export.table.toDrive task per chunk (avoids the 10 MB GEE payload
         limit that is triggered when a single large FeatureCollection is
         serialised into the export-request body)
      5. Optionally monitor all tasks until completion

    Output CSVs land in Google Drive under drive_folder.
    Files with >POINT_CHUNK_SIZE points produce multiple CSVs named
      extracted_<stem>_part000.csv, extracted_<stem>_part001.csv, ...
    Merge them afterwards with:
      pd.concat([pd.read_csv(f) for f in sorted(glob('extracted_<stem>_part*.csv'))])
    """
    print(f"\n{'='*70}")
    print(f'ECHOSOUNDER EXTRACTION — {len(bounds_df)} files')
    print(f'Bands        : {rrs_bands}')
    print(f'Chunk size   : {POINT_CHUNK_SIZE} pts/task')
    print(f'Drive folder : {drive_folder}')
    print(f"{'='*70}")

    all_tasks = []

    for _, row in bounds_df.iterrows():
        fname = row['filename']
        fpath = os.path.join(gpkg_dir, fname)
        stem  = fname.replace('.gpkg', '')

        print(f"\n{'─'*70}")
        print(f'  {fname}  ({row["n_points"]} points)')

        # ── AOI: point bounding box + 0.01° buffer (~1 km) ───────────────
        buf      = 0.01
        aoi_geom = ee.Geometry.Rectangle(
            [row['min_lon'] - buf, row['min_lat'] - buf,
             row['max_lon'] + buf, row['max_lat'] + buf],
            proj='EPSG:4326', geodesic=False,
        )
        print(f'  AOI: ({row["min_lon"]:.4f}, {row["min_lat"]:.4f}) '
              f'→ ({row["max_lon"]:.4f}, {row["max_lat"]:.4f})')

        # ── Full S2 scenes intersecting AOI ───────────────────────────────
        l1c      = get_l1c_collection(aoi_geom, start_date, end_date, max_cloud_cover)
        n_scenes = l1c.size().getInfo()
        if n_scenes == 0:
            print(f'  ✗ No valid scenes — skipping')
            continue
        print(f'  {n_scenes} scenes — running ACOLITE DSF on full scenes...')

        # ── ACOLITE DSF on full scenes → clip to AOI → mean composite ────
        composite, n_ok = build_rrs_composite(l1c, aoi_geom)
        if composite is None:
            print(f'  ✗ All scenes failed DSF — skipping')
            continue
        print(f'  Composite: {n_ok}/{n_scenes} scenes ✓')

        # ── Read echosounder points ────────────────────────────────────────
        gdf = gpd.read_file(fpath)
        gdf.columns = [c.lower() for c in gdf.columns]
        if 'lon' not in gdf.columns and gdf.geometry is not None:
            gdf['lon'] = gdf.geometry.x
            gdf['lat'] = gdf.geometry.y
        gdf = gdf.reset_index(drop=True)

        # ── Chunked export (avoids 10 MB GEE payload limit) ──────────────
        n_chunks = -(-len(gdf) // POINT_CHUNK_SIZE)   # ceiling division
        print(f'  Exporting {len(gdf)} points in {n_chunks} chunk(s)...')
        chunk_tasks = start_chunked_export_tasks(
            composite    = composite,
            gdf          = gdf,
            rrs_bands    = rrs_bands,
            base_name    = f'extracted_{stem}',
            drive_folder = drive_folder,
            scale        = scale,
        )
        all_tasks.extend(chunk_tasks)

    # ── Summary ───────────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    print(f'Tasks started : {len(all_tasks)}')
    print(f'Drive folder  : {drive_folder}')
    print(f'Monitor at    : https://code.earthengine.google.com/tasks')
    print(f"{'='*70}")
    for t in all_tasks:
        print(f"  → {t['name']}.csv")

    # ── Optionally monitor until all tasks complete ────────────────────────
    if monitor and all_tasks:
        monitor_tasks(all_tasks, check_interval=check_interval)

    return all_tasks


# =========================
# RUN
# =========================

tasks = process_all_gpkg(
    gpkg_dir        = gpkg_dir,
    bounds_df       = bounds_df,
    start_date      = start_date,
    end_date        = end_date,
    rrs_bands       = RRS_BANDS,
    max_cloud_cover = 30,
    scale           = 10,
    drive_folder    = drive_folder,
    monitor         = True,       # set False to skip waiting
    check_interval  = 30,
)



ECHOSOUNDER EXTRACTION — 3 files
Bands        : ['B1', 'B2', 'B3', 'B4', 'B8', 'B8A', 'B11', 'B12']
Chunk size   : 500 pts/task
Drive folder : ECHOSOUNDER

──────────────────────────────────────────────────────────────────────
  bathymetry_masinloc_dry2024.gpkg  (1356 points)
  AOI: (119.8992, 15.5049) → (119.9467, 15.5516)
  20 scenes — running ACOLITE DSF on full scenes...
<---------------------------------------------------------------------------------------------------->
Pdark: {'B1': 0.08289999999999999, 'B2': 0.0025, 'B3': 0.0034, 'B4': 0, 'B5': 0, 'B6': 0, 'B7': 0.000125, 'B8': 0, 'B8A': 0, 'B9': 0.001800000000000001, 'B10': 0, 'B11': 0, 'B12': 0}
Geometry: sza=41.7999940593363, vza=8.841800087219859, raa=136.05539073876636
Gases: uwv=1.5, uoz=0.3, pressure=1013.25
Lut: sel_lut='ACOLITE-LUT-202110-MOD1', sel_aot=np.float64(0.001)
<---------------------------------------------------------------------------------------------------->
    scene 1/20: ✓ DSF done
<------------------

<---------------------------------------------------------------------------------------------------->
Pdark: {'B1': 0.12036666666666607, 'B2': 0.08535833333333333, 'B3': 0.06204886363636359, 'B4': 0.038921428571428565, 'B5': 0.0267, 'B6': 0.0485, 'B7': 0.0474, 'B8': 0.027, 'B8A': 0.0402, 'B9': 0.0031000000000000016, 'B10': 0.0005000000000000003, 'B11': 0, 'B12': 0}
Geometry: sza=19.0920626836852, vza=8.92465437008476, raa=178.85202379828058
Gases: uwv=1.5, uoz=0.3, pressure=1013.25
Lut: sel_lut='ACOLITE-LUT-202110-MOD1', sel_aot=np.float64(0.001)
<---------------------------------------------------------------------------------------------------->
    scene 13/20: ✓ DSF done
<---------------------------------------------------------------------------------------------------->
Pdark: {'B1': 0.0806, 'B2': 0.0036, 'B3': 0.0047, 'B4': 0, 'B5': 0, 'B6': 0.01705, 'B7': 0, 'B8': 0.0112, 'B8A': 0, 'B9': 0.003825000000000011, 'B10': 0.00019999999999999987, 'B11': 0.004690909090909091, 'B12': 0

<---------------------------------------------------------------------------------------------------->
Pdark: {'B1': 0.10769999999999993, 'B2': 0.0682, 'B3': 0.0457, 'B4': 0.025156249999999998, 'B5': 0.0165, 'B6': 0.0031, 'B7': 0.0098, 'B8': 0.012, 'B8A': 0.006566666666666668, 'B9': 0.004299999999999999, 'B10': 0.0006000000000000017, 'B11': 0.0056, 'B12': 0.0013}
Geometry: sza=35.0487062090012, vza=10.668318690584169, raa=27.286725648606946
Gases: uwv=1.5, uoz=0.3, pressure=1013.25
Lut: sel_lut='ACOLITE-LUT-202110-MOD1', sel_aot=np.float64(0.001)
<---------------------------------------------------------------------------------------------------->
    scene 94/403: ✓ DSF done
<---------------------------------------------------------------------------------------------------->
Pdark: {'B1': 0.11109999999999999, 'B2': 0.0799, 'B3': 0.04653370786516855, 'B4': 0.0263, 'B5': 0.021242857142857136, 'B6': 0.019399999999999997, 'B7': 0.017314285714285706, 'B8': 0.013528125000000007, 'B8A': 0.0

<---------------------------------------------------------------------------------------------------->
Pdark: {'B1': 0.09699999999999999, 'B2': 0.07186666666666668, 'B3': 0.0445375, 'B4': 0.029, 'B5': 0.0249, 'B6': 0.0231, 'B7': 0.0208, 'B8': 0.017099999999999997, 'B8A': 0.01915, 'B9': 0.004299999999999999, 'B10': 0, 'B11': 0.010793659420289824, 'B12': 0.007336538461538445}
Geometry: sza=25.7842072206093, vza=8.435708594883605, raa=163.60151562671265
Gases: uwv=1.5, uoz=0.3, pressure=1013.25
Lut: sel_lut='ACOLITE-LUT-202110-MOD2', sel_aot=np.float64(0.18180391055252143)
<---------------------------------------------------------------------------------------------------->
    scene 205/403: ✓ DSF done
<---------------------------------------------------------------------------------------------------->
Pdark: {'B1': 0.09492000000000016, 'B2': 0.0445, 'B3': 0.0377, 'B4': 0.0039, 'B5': 0.023276666666666716, 'B6': 0.013950000000000002, 'B7': 0.0132, 'B8': 0.01844565648552123, 'B8A': 0.0025

<---------------------------------------------------------------------------------------------------->
Pdark: {'B1': 0.10837999999999962, 'B2': 0.057, 'B3': 0.044495302013422854, 'B4': 0.0199, 'B5': 0.0189, 'B6': 0.0074, 'B7': 0.0048, 'B8': 0.01065, 'B8A': 0.0029, 'B9': 0.0035500000000000024, 'B10': 0.0003999999999999988, 'B11': 0.0015, 'B12': 0}
Geometry: sza=21.5088562937744, vza=9.82404543535665, raa=31.501717258253592
Gases: uwv=1.5, uoz=0.3, pressure=1013.25
Lut: sel_lut='ACOLITE-LUT-202110-MOD1', sel_aot=np.float64(0.001)
<---------------------------------------------------------------------------------------------------->
    scene 307/403: ✓ DSF done
<---------------------------------------------------------------------------------------------------->
Pdark: {'B1': 0.10756666666666671, 'B2': 0.060899999999999996, 'B3': 0.0435, 'B4': 0.0043, 'B5': 0.018866666666666667, 'B6': 0.0003, 'B7': 0.0142, 'B8': 0.0107, 'B8A': 0.0041, 'B9': 0.003700000000000002, 'B10': 0.00039999999999999

EEException: Request payload size exceeds the limit: 10485760 bytes.